# Imputing Missing Mutational Variant-Effect Scores: Benchmarking Protein Language Models and Interpretable Biological Features

## Introduction
Deep mutational scanning (DMS) measures the functional effects of many protein variants at once, but resulting sequence-function maps are rarely complete. This repository provides a pipeline (produced for the purposes of MSc dissertation), for benchmarking methods to impute missing scores. Here, encoders turn variant sequences into numerical representations, and regressors map representations to effects.

Encoders currently supported:
- ESM-1, ESM-1b, ESM-1v and ESM-2 (FAIR)
- ESM-C and ESM-3 open (EvolutionaryScale)
- ProtBERT and ProtT5

Hand-crafted representations implemented:
- One-hot variant descriptor
- BOHN - introduced combining BLOSUM62 (B) substitution scores, one-hot (OH) variant descriptors, with a naïve (N) within-fold positional-mean feature (fall-back to overall mean)

Regressors currently implemented span linear, regularised linear, kernel, tree, ensemble and instance-based families.

## Environment Setup

`fair-esm` and EvolutionaryScale's `esm` are both imported under `esm` and cannot be installed together. Encoders from the two families must therefore be run in separate environments or sequentially.

GPU-acceleration via CUDA is optional, though it makes encoding much faster.

In [ ]:
# Importing necessary libraries and functions
import time # Time keeping
import gc # Garbage collection
import os # Utilities
import numpy as np # Numerical operations
import matplotlib.pyplot as plt # Plotting
import pandas as pd # Data manipulation
import torch # PyTorch machine learning
from sklearn.model_selection import KFold # Data splits and cross-validation
from sklearn.preprocessing import StandardScaler # Numerical data scaling
from sklearn.metrics import r2_score, mean_squared_error # Regression scoring metrics
from sklearn.linear_model import LinearRegression, Lasso, ElasticNet, SGDRegressor # Linear regressors
from sklearn.cross_decomposition import PLSRegression # Partial least squares regressor
from sklearn.gaussian_process import GaussianProcessRegressor, kernels as GPK # Gaussian process regressor
from sklearn.kernel_ridge import KernelRidge # Kernel ridge regressor
from sklearn.svm import SVR # Support vector regressor
from sklearn.neighbors import KNeighborsRegressor # K-nearest neighbours regressor
from sklearn.tree import DecisionTreeRegressor # Decision tree regressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, AdaBoostRegressor, GradientBoostingRegressor # Ensemble regressors
import warnings # Controlling warnings
warnings.filterwarnings("ignore")

# Importing from created .py files
import data_loader # Loads data
import naive_baseline # Imputes scores using positional mutational averages
import onehot_encoder # Encodes proteins with one-hot, saves/loads embedding datasets
import blosum as blosum # Assesses BLOSUM substitution scores against functional scores
import bohn # Encodes proteins with BOHN, saves/loads embedding datasets
import esm_encoder as esm_encoder # Embeds proteins with ESM, saves/loads embedding datasets
import esmc_encoder as esmc_encoder # Embeds proteins with ESM-C, saves/loads embedding datasets
import esm3_encoder # Embeds proteins with ESM-3, saves/loads embedding datasets
import prott5_encoder # Embeds proteins with ProtT5, saves/loads embedding datasets
import protbert_encoder # Embeds proteins with ProtBert, saves/loads embedding datasets
import sequence_function_heatmap # Plots sequence-function heatmaps
import hyperparameter_tuning # Tunes hyperparameters given a grid and model

In [ ]:
# Setting seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# Selecting device for PyTorch and displaying installation info
print(f"PyTorch version: {torch.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # Checking CUDA availability, with CPU as fallback
print(f"Using device: {device}") # Printing which device isin use
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}") # Printing GPU name if in use

## Input Format

Each protein dataset should be a folder containing three files:

- `all_variants.fasta` - single-residue variants (e.g. 'M1A'): descriptors and full sequences (clean this file if removing outliers)
- `scores.csv` - measured functional scores, one row per scored variant (clean this file if removing outliers)
- `wild_type.fasta` - reference sequence

Running a benchmark produces:

- `<protein>_<encoder>_embeddings.pkl` - cached embeddings, reused on later runs
- `<protein>_results.csv` - appends encoder-regressor pair results

Both output files are written to the working directory rather than the dataset folder.

In [ ]:
# Example
datasets = ["HRAS", "P53", "PTEN"] # Specify protein dataset folder names

## Naïve Positional Baselines and BLOSUM62

In [ ]:
for dataset in datasets:
    naive_baseline.runNaiveBaseline(dataset) # Getting results using naive biological baseline
    blosum.assessBlosum(dataset) # Assessing BLOSUM scores

In [ ]:
# Defining function to fit and evaluate regressors
def runRegressor(X_train, X_test, y_train, y_test, regressor, param_grid=None, grid_scoring="neg_root_mean_squared_error"):
    best_params = None # Setting default best parameters to none

    # Checking if parameter grid supplied
    if param_grid is not None:
        regressor, best_params = hyperparameter_tuning.tuneHyperparameters(regressor, param_grid, X_train, y_train, scoring=grid_scoring) # Tuning hyperparameters
    else:
        regressor.fit(X_train, y_train) # Fitting model with default parameters

    y_pred = regressor.predict(X_test) # Making predictions
    r2 = r2_score(y_test, y_pred) # Calculating R-squared score
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))

    # Displaying results
    print(f"{type(regressor).__name__} R2 score: {r2:.5f}") 
    print(f"{type(regressor).__name__} RMSE: {rmse:.5f}")
    print("\n" + "="*50) # Results separator

    return regressor, best_params, r2, rmse

# Defining function to run a whole experiment, including generating embeddings, preprocessing, fine-tuning, training, and evaluation
def runExperiment(dataset, protein_language_model, regressor_raw, device, param_grid=None, batch_size=128, scale_data=True,
                  grid_scoring="neg_root_mean_squared_error", verbose=False, embed_only=False, half=False,
                  use_esmc=False, use_esm3=False, use_prott5=False, use_protbert=False):
    print("="*50 + "\nExperiment details:")
    print(f"\tDataset: {dataset}")
    print(f"\tProtein language model: {protein_language_model}")
    print(f"\tBase regressor: {type(regressor_raw).__name__}")
    print(f"\tStandard scaling features: {scale_data}")
    if param_grid is not None: print(f"\tOptimising hyperparameters with: {grid_scoring}\n" + "-"*50)
    else: print("\tOptimising hyperparameters: False\n" + "-"*50)

    encode_start = time.time() # Marking encode start

    save_path = f"{dataset}_{protein_language_model}_embeddings.pkl" # Expected embedding file path

    # Loading embeddings if they exist
    if os.path.exists(save_path):
        full_df = pd.read_pickle(save_path) # Loading embeddings
        esm_model = None # Not loading encoder
    elif protein_language_model == "onehot":
        full_df = onehot_encoder.generateEmbeddingDataset(dataset, verbose=verbose) # Generating one-hot features
        esm_model = None # No encoder for one-hot
    elif protein_language_model == "bohn":
        full_df = bohn.generateEmbeddingDataset(dataset, verbose=verbose) # Generating BOHN features
        esm_model = None # No encoder for BOHN
    elif use_esmc == True:
        esm_model = esmc_encoder.loadEncoder(protein_language_model, device, verbose) # Loading ESM-C model
        full_df = esmc_encoder.generateEmbeddingDataset(dataset, protein_language_model, esm_model, device, verbose=verbose) # Generating ESM-C embeddings
    elif use_esm3 == True:
        esm_model = esm3_encoder.loadEncoder(protein_language_model, device, verbose) # Loading ESM-3 model
        full_df = esm3_encoder.generateEmbeddingDataset(dataset, protein_language_model, esm_model, device, verbose=verbose) # Generating ESM-3 embeddings
    elif use_prott5 == True:
        esm_model, tokenizer = prott5_encoder.loadEncoder(protein_language_model, device, verbose) # Loading ProtT5 encoder
        full_df = prott5_encoder.generateEmbeddingDataset(dataset, protein_language_model, esm_model, tokenizer, device, verbose=verbose) # Generating ProtT5 embeddings
    elif use_protbert == True:
        esm_model, tokenizer = protbert_encoder.loadEncoder(protein_language_model, device, verbose) # Loading ProtBERT encoder
        full_df = protbert_encoder.generateEmbeddingDataset(dataset, protein_language_model, esm_model, tokenizer, device, verbose=verbose) # Generating ProtBERT embeddings
    else:
        esm_model, alphabet, batch_converter = esm_encoder.loadEncoder(protein_language_model, device, verbose, half=half) # Loading ESM model
        full_df = esm_encoder.generateEmbeddingDataset(dataset, protein_language_model, esm_model, alphabet, batch_converter, device, batch_size, verbose=verbose) # Generating ESM embeddings

    encode_time = time.time() - encode_start # Measuring encoding time

    # Stopping after embedding generation if only caching embeddings for later
    if embed_only == True:
        print(f"Embeddings ready for {protein_language_model} ({encode_time:.2f}s)") # Signalling completion

        # Dropping references to loaded encoder
        if esm_model is not None:
            del esm_model # Dropping encoder reference if one was loaded
        if "batch_converter" in locals():
            del batch_converter # Dropping batch converter if one was created

        gc.collect() # Forcing Python to release unreferenced objects
        if device.type == "cuda":
            torch.cuda.empty_cache() # Releasing cached GPU memory back to the driver

        return # Exiting before training
    
    feature_columns = [col for col in full_df.columns if col.startswith("feature_")] # Getting embedded feature column names
    
    known_df = full_df[full_df["score"].notna()] # Known variants with scores (for training and validation)
    X_known = known_df[feature_columns].values # Setting features
    y_known = known_df["score"].values # Setting target
    
    kfold = KFold(n_splits=5, shuffle=True, random_state=42) # Setting up reproducible 5-fold cross-validation

    r2_scores = [] # Initialising list to store fold R-squared scores
    rmse_scores = [] # Initialising list to store fold RMSE scores

    train_start = time.time() # Marking training start

    # Iterating through folds
    for fold, (train_idx, test_idx) in enumerate(kfold.split(X_known), start=1):
        print(f"Fold {fold}:") # Signalling current fold
        X_train, X_test = X_known[train_idx], X_known[test_idx] # Splitting features for fold
        y_train, y_test = y_known[train_idx], y_known[test_idx] # Splitting targets for fold

        # Fitting scaling only on training data so test data doesn't leak
        if scale_data == True:
            scaler = StandardScaler() # Calling standard scaler

            if protein_language_model == "onehot":
                pass # One-hot features are binary so no scaling needed

            elif protein_language_model == "bohn":
                n_blosum = 41 # BLOSUM block: 1 substitution score + 20 WT row + 20 mutant row
                n_onehot = X_train.shape[1] - n_blosum # One-hot is everything before the BLOSUM block

                X_train_cont = scaler.fit_transform(X_train[:, n_onehot:]) # Scaling only continuous features
                X_test_cont = scaler.transform(X_test[:, n_onehot:]) # Transforming test features

                X_train = np.hstack([X_train[:, :n_onehot], X_train_cont]) # Recombining unscaled one-hot with scaled continuous
                X_test = np.hstack([X_test[:, :n_onehot], X_test_cont]) # Recombining unscaled one-hot with scaled continuous

            else:
                X_train = scaler.fit_transform(X_train) # Fitting and scaling training features
                X_test = scaler.transform(X_test) # Transforming test features

        # Adding per-fold positional mean as a feature (computed from training fold only to avoid leakage)
        if protein_language_model == "bohn":
            positions_known = known_df["variant"].str[1:-1].astype(int).values # Position of each known variant, for per-fold means
            pos_train, pos_test = positions_known[train_idx], positions_known[test_idx] # Positions for this fold
            train_means = pd.DataFrame({"position": pos_train, "score": y_train}).groupby("position")["score"].mean() # Mean score per-position from training only
            global_mean = y_train.mean() # Fallback for positions unseen in training

            train_mean_feat = np.array([train_means.get(p, global_mean) for p in pos_train]).reshape(-1, 1) # Positional mean for train variants
            test_mean_feat = np.array([train_means.get(p, global_mean) for p in pos_test]).reshape(-1, 1) # Positional mean for test variants

            mean_scaler = StandardScaler() # Scaling the positional average on its own
            train_mean_feat = mean_scaler.fit_transform(train_mean_feat) # Fitting and scaling on training only
            test_mean_feat = mean_scaler.transform(test_mean_feat) # Transforming test values

            X_train = np.hstack([X_train, train_mean_feat]) # Appending scaled positional means to train features
            X_test = np.hstack([X_test, test_mean_feat]) # Appending scaled means to test features

        regressor, best_params, r2, rmse = runRegressor(X_train, X_test, y_train, y_test, regressor_raw, param_grid=param_grid, grid_scoring=grid_scoring) # Running regressor on fold
        r2_scores.append(r2) # Storing fold R-squared
        rmse_scores.append(rmse) # Storing fold RMSE

    train_time = time.time() - train_start # Measuring training time across all folds

    # Displaying averaged cross-validation results
    print(f"Mean R2 score: {np.mean(r2_scores):.5f} (+/- {np.std(r2_scores):.5f})")
    print(f"Mean RMSE: {np.mean(rmse_scores):.5f} (+/- {np.std(rmse_scores):.5f})")
    print(f"Encoding time: {encode_time:.2f}s") # Displaying encode duration
    print(f"Total 5-fold CV training time: {train_time:.2f}s") # Displaying training duration
    print("\n" + "="*50) # Results separator

    # Dropping references to loaded encoder
    if esm_model is not None:
        del esm_model # Dropping encoder reference if one was loaded
    if "batch_converter" in locals():
        del batch_converter # Dropping batch converter if one was created

    gc.collect() # Forcing Python to release unreferenced objects
    if device.type == "cuda":
        torch.cuda.empty_cache() # Releasing cached GPU memory back to the driver

    return {"dataset": dataset,
            "encoder": protein_language_model,
            "regressor": type(regressor_raw).__name__,
            "mean_r2": np.mean(r2_scores),
            "std_r2": np.std(r2_scores),
            "mean_rmse": np.mean(rmse_scores),
            "std_rmse": np.std(rmse_scores),
            "encode_time": encode_time,
            "train_time": train_time,
            "folds_r2": [float(x) for x in r2_scores],
            "folds_rmse": [float(x) for x in rmse_scores]}

# Defining function to run defined encoders alongside defined regressors and save results
def runBenchmark(dataset, encoders, regressors):
    # Loading existing results to avoid re-runs
    if os.path.exists(f"{dataset}_results.csv"):
        results_df = pd.read_csv(f"{dataset}_results.csv") # Loading previous results
        all_results = results_df.to_dict("records") # Converting back to list of dictionaries
        done = set(zip(results_df["encoder"], results_df["regressor"])) # Completed pairs
    else:
        all_results = []
        done = set()

    # Running experiments varying encoders and regressors
    for encoder, run, use_esmc, use_esm3, use_prott5, use_protbert in encoders:
        device = torch.device(run) # Selecting device
        for regressor, param_grid in regressors:
            # Skipping already completed pairs
            if (encoder, type(regressor).__name__) in done:
                print(f"Skipping {encoder} / {type(regressor).__name__} (already done)")
                continue
            result = runExperiment(dataset, encoder, regressor, device, param_grid, use_esmc=use_esmc, use_esm3=use_esm3, use_prott5=use_prott5, use_protbert=use_protbert) # Running experiment
            if result is not None:
                all_results.append(result) # Storing result
            pd.DataFrame(all_results).to_csv(f"{dataset}_results.csv", index=False) # Saving results

    print(f"Saved {len(all_results)} results to {dataset}_results.csv")

## Running Benchmarks (Implemented Models)

### Hand-crafted
| Calling String | Category |
|---|---|
| onehot | Hand-crafted |
| bohn | Hand-crafted |

### ESM (FAIR)
| Calling String | Category |
|---|---|
| esm1_t6_43M_UR50S | ESM-1 |
| esm1_t12_85M_UR50S | ESM-1 |
| esm1_t34_670M_UR50S | ESM-1 |
| esm1_t34_670M_UR50D | ESM-1 |
| esm1_t34_670M_UR100 | ESM-1 |
| esm1b_t33_650M_UR50S | ESM-1b |
| esm1v_t33_650M_UR90S_1 | ESM-1v |
| esm1v_t33_650M_UR90S_2 | ESM-1v |
| esm1v_t33_650M_UR90S_3 | ESM-1v |
| esm1v_t33_650M_UR90S_4 | ESM-1v |
| esm1v_t33_650M_UR90S_5 | ESM-1v |
| esm2_t6_8M_UR50D | ESM-2 |
| esm2_t12_35M_UR50D | ESM-2 |
| esm2_t30_150M_UR50D | ESM-2 |
| esm2_t33_650M_UR50D | ESM-2 |
| esm2_t36_3B_UR50D | ESM-2 |
| esm2_t48_15B_UR50D | ESM-2 |

### ESM (EvolutionaryScale)
| Calling String | Category |
|---|---|
| esmc_300m | ESM-C |
| esmc_600m | ESM-C |
| esm3-open | ESM-3 |

ESM-C 6B is publicly available but was not benchmarked here, as the model exceeded available memory, though it should work on the module given sufficient RAM.

ESM-3 7B and ESM-3 98B are available only through the EvolutionaryScale API and are not supported by this pipeline.

### ProtTrans Models
| Calling String | Category |
|---|---|
| prot_bert | ProtBert |
| prot_t5_xl_half_uniref50-enc | ProtT5 |

In [ ]:
# Encoders: [name, device, use_esmc, use_esm3, use_prott5, use_protbert]
encoders = [
    ["onehot", "cpu", False, False, False, False],
    ["bohn", "cpu", False, False, False, False],
    ["esm1_t6_43M_UR50S", "cuda", False, False, False, False],
    ["esm1_t12_85M_UR50S", "cuda", False, False, False, False],
    ["esm1_t34_670M_UR50S", "cuda", False, False, False, False],
    ["esm1_t34_670M_UR50D", "cuda", False, False, False, False],
    ["esm1_t34_670M_UR100", "cuda", False, False, False, False],
    ["esm1b_t33_650M_UR50S", "cuda", False, False, False, False],
    ["esm1v_t33_650M_UR90S_1", "cuda", False, False, False, False],
    ["esm1v_t33_650M_UR90S_2", "cuda", False, False, False, False],
    ["esm1v_t33_650M_UR90S_3", "cuda", False, False, False, False],
    ["esm1v_t33_650M_UR90S_4", "cuda", False, False, False, False],
    ["esm1v_t33_650M_UR90S_5", "cuda", False, False, False, False],
    ["esm2_t6_8M_UR50D", "cuda", False, False, False, False],
    ["esm2_t12_35M_UR50D", "cuda", False, False, False, False],
    ["esm2_t30_150M_UR50D", "cuda", False, False, False, False],
    ["esm2_t33_650M_UR50D", "cuda", False, False, False, False],
    ["esm2_t36_3B_UR50D", "cpu", False, False, False, False],
    ["esm2_t48_15B_UR50D", "cpu", False, False, False, False],
    ["esmc_300m", "cuda", True, False, False, False],
    ["esmc_600m", "cuda", True, False, False, False],
    ["esm3-open", "cpu", False, True, False, False],
    ["prot_bert", "cpu", False, False, False, True],
    ["prot_t5_xl_half_uniref50-enc", "cpu", False, False, True, False]
]

# PLS class to solve PLSRegression predict() output issue
class PLSRegressor(PLSRegression):
    def __init__(self, n_components=2):
        super().__init__(n_components=n_components)
    def predict(self, X):
        return super().predict(X).flatten()
    
# Regressor parameter grids
param_grid_lin = {}
param_grid_las = {"alpha": [1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 1]}
param_grid_pls = {"n_components": [2, 4, 8, 16, 32, 64]}
param_grid_gpr = {"kernel": [GPK.ConstantKernel(), GPK.DotProduct(), GPK.ExpSineSquared(),
                             GPK.Matern(), GPK.RBF(), GPK.RationalQuadratic(), GPK.WhiteKernel()]}
param_grid_ent = {"alpha": [1e-05, 0.0001, 0.001, 0.01, 0.1, 1],
                  "l1_ratio": [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]}
param_grid_krr = {"kernel": ["linear", "poly", "polynomial", "rbf", "laplacian", "sigmoid", "cosine"],
                  "alpha": [0.003, 0.01, 0.03, 0.1, 0.3, 1, 3]}
param_grid_svr = {"epsilon": [1e-09, 1e-08, 1e-07, 1e-06, 1e-05, 1e-04, 0.001, 0.01, 0.1, 1],
                  "C": [0.01, 0.03, 0.1, 0.3, 1, 3, 10]}
param_grid_knn = {"weights": ["uniform", "distance"],
                  "n_neighbors": [4, 5, 6, 7, 8, 9, 10, 11]}
param_grid_sgd = {"loss": ["squared_error", "huber", "epsilon_insensitive", "squared_epsilon_insensitive"],
                  "penalty": ["l2", "l1", "elasticnet"],
                  "alpha": [1e-05, 3e-05, 0.0001, 0.0003, 0.001, 0.003, 0.01]}
param_grid_dtr = {"min_samples_split": [2, 6, 20, 60, 200, 600],
                  "min_samples_leaf": [1, 3, 10, 30, 100, 300],
                  "min_weight_fraction_leaf": [0, 1e-05, 0.0001, 0.001, 0.01, 0.1]}
param_grid_rfr = {"n_estimators": [100, 200],
                  "min_samples_leaf": [1, 2, 4, 8],
                  "max_features": [0.3, 0.5, 0.7, 1]}
param_grid_xtr = {"n_estimators": [100, 200, 300],
                  "min_samples_leaf": [1, 2, 4, 8]}
param_grid_ada = {"estimator__max_depth": [3, 5, 10, 20],
                  "estimator__min_samples_leaf": [1, 2, 4, 8]}
param_grid_gbr = {"n_estimators": [100, 200, 300],
                  "learning_rate": [0.01, 0.1],
                  "max_features": [0.2, 0.5, 1]}

# Regressor models and associated grids
regressors = [
    [LinearRegression(), param_grid_lin],
    [Lasso(), param_grid_las],
    [PLSRegressor(), param_grid_pls],
    [GaussianProcessRegressor(), param_grid_gpr],
    [ElasticNet(), param_grid_ent],
    [KernelRidge(), param_grid_krr],
    [SVR(), param_grid_svr],
    [KNeighborsRegressor(), param_grid_knn],
    [SGDRegressor(), param_grid_sgd],
    [DecisionTreeRegressor(random_state=42), param_grid_dtr],
    [RandomForestRegressor(random_state=42), param_grid_rfr],
    [ExtraTreesRegressor(random_state=42), param_grid_xtr],
    [AdaBoostRegressor(estimator=DecisionTreeRegressor(), random_state=42), param_grid_ada],
    [GradientBoostingRegressor(random_state=42), param_grid_gbr]
]

In [ ]:
# Example:
for dataset in datasets:
    runBenchmark(dataset, encoders, regressors)